In [0]:
import pyspark.sql.functions as f
from pyspark.sql.types import StringType, IntegerType, DateType, FloatType

catalog_name = "ecommerce"

# Dimension Table: Brand

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_brand")
display(df_bronze.limit(10))

In [0]:
df_silver = df_bronze.withColumn('brand_name', f.trim(f.col('brand_name')))
display(df_silver.limit(10))

In [0]:
df_silver = df_silver.withColumn(
    'brand_code',
    f.regexp_replace(f.col('brand_code'), '[^A-Za-z0-9]', '')
)
display(df_silver.limit(10))

In [0]:
df_silver.select("category_code").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "category_code",
    f.when(f.col("category_code") == "GROCERY", "GRCY")
     .when(f.col("category_code") == "TOYS", "TOY")
     .when(f.col("category_code") == "BOOKS", "BKS")
     .otherwise(f.col("category_code"))
)
display(df_silver.select("category_code").distinct())

In [0]:
df_silver.write.format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.slv_brand')

# Dimension Table: Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
display(df_bronze.limit(10))

In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("category_code")
df_duplicates = df_bronze.withColumn("count", f.count("category_code").over(window_spec)) \
    .filter(f.col("count") > 1)

display(df_duplicates.select("category_code", "count").distinct())
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(["category_code"])
display(df_silver)

In [0]:
df_silver = df_silver.withColumn(
    "category_code",
    f.upper(f.col("category_code"))
)
display(df_silver)

In [0]:
df_silver.write.format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.slv_category')

# Dimension Table: Customers

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_customers") 
display(df_bronze.limit(10))

In [0]:
df_silver = df_bronze.withColumn(
    "phone",
    f.regexp_replace(f.col("phone"), r"\.0$", "")
)
display(df_silver)

In [0]:
df_silver = df_silver.withColumn("phone_missing_flag",
    f.when(f.col("phone").isNull(), True).otherwise(False)
)
display(df_silver.limit(10))

In [0]:
df_silver = df_silver.fillna({
    'phone': 'Not Available'
})
df_silver.filter(f.col('phone').isNull()).show()

In [0]:
df_silver.filter(f.col('customer_id').isNull()).show(3)

In [0]:
df_silver = df_silver.dropna(subset=['customer_id'])
count_of_row = df_silver.count()
print(f'The row count after deleting the null customer_id is {count_of_row}')


In [0]:
df_silver.filter(f.col('customer_id').isNull()).show(3)


In [0]:
df_silver.write.format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.slv_customers')

# Dimension Table: products

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_products")
display(df_bronze.limit(10))




In [0]:
null_counts = df_bronze.select([f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df_bronze.columns])
display(null_counts)


In [0]:
df_silver = df_bronze.fillna({
    'color': 'No Colour'
})
df_silver.filter(f.col('color').isNull()).show(3)



In [0]:
df_silver = df_silver.withColumn(
    "weight_grams",
    f.regexp_replace(f.col("weight_grams"), "g", "")
)
df_silver.select(f.col("weight_grams")).show(10)


In [0]:
df_silver.select('length_cm').show(5)

In [0]:
df_silver = df_silver.withColumn(
    "length_cm",
    f.regexp_replace(f.col("length_cm"), ",", ".")
)
df_silver.select(f.col("length_cm")).show(5)

In [0]:
df_silver = df_silver.withColumn(
    "category_code",
    f.upper(f.col("category_code"))
).withColumn(
    "brand_code",
    f.upper(f.col("brand_code"))
)
display(df_silver.limit(10))

In [0]:
df_silver.select("material").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "material",
    f.when(f.col("material") == "Coton", "Cotton")
     .when(f.col("material") == "Alumium", "Aluminum")
     .when(f.col("material") == "Ruber", "Rubber")
     .otherwise(f.col("material"))
)
df_silver.select("material").distinct().show()    

In [0]:
df_silver.filter(f.col("rating_count") < 0).select("rating_count").show(10)

In [0]:
df_silver = df_silver.withColumn(
    "rating_count",
    f.when(f.col("rating_count").isNotNull(), f.abs(f.col("rating_count")))
     .otherwise(f.lit(0))  # if null, replace with 0
)

df_silver.filter(f.col("rating_count") < 0).select("rating_count").show(10)

In [0]:
display(df_silver.limit(10))

In [0]:
df_silver.write.format('delta') \
    .mode('overwrite') \
    .option('mergeSchema', 'true') \
    .saveAsTable(f'{catalog_name}.silver.slv_products')

# Dimension Table: Date

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(3)

In [0]:
# Read the raw data from the bronze table (ecommerce.bronze.brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calendar")

# Get row and column count
row_count, column_count = df_bronze.count(), len(df_bronze.columns)

# Print the results
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")

df_bronze.show(3)

In [0]:
# Remove duplicate rows
df_silver = df_bronze.dropDuplicates(['date'])

# Get row count
row_count = df_silver.count()

print("Rows After removing Duplicates: ", row_count)

In [0]:
# Capitalize first letter of each word in day_name
df_silver = df_silver.withColumn("day_name", f.initcap(f.col("day_name")))

df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn("week_of_year", f.abs(f.col("week_of_year")))  # Convert negative to positive

df_silver.show(3)

In [0]:
df_silver = df_silver.withColumn("quarter", f.concat_ws("", f.concat(f.lit("Q"), f.col("quarter"), f.lit("-"), f.col("year"))))

df_silver = df_silver.withColumn("week_of_year", f.concat_ws("-", f.concat(f.lit("Week"), f.col("week_of_year"), f.lit("-"), f.col("year"))))

df_silver.show(3)

In [0]:
df_silver = df_silver.withColumnRenamed("week_of_year", "week")

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.silver.slv_calendar")